# 05 Matrix imaging

For each scenario from 04: phase 1 images the BDT with the reconnaissance grid and picks where the BDT high is,
then the rolling phase 2 patches are centred there and image it. The static designs are run for comparison.
Each image is scored on how much of the BDT top it finds (coverage), the depth error, and how far the picked BDT
high is from the true one. Noise-free, and focused with the same 3D model that made the echoes.

The forward model, imaging and scoring are in `mi_common.py`, shared with 05b, 06 and 07:

- **Earth:** the background model of 01 plus the low-velocity zone (LVZ) of one 04 scenario, as Born scatterers.
- **Echoes:** reflection matrix R(f) between every pair of stations that record at the same time.
- **Image:** confocal image, focused with the 3D travel-time tables from 03.
- **Score:** a column is imaged if its peak is `SNR_MIN` times the image's median; it covers the top if its picked
  top is within `TOP_TOL` km of the truth.

Inputs: `data/velocity_model.npz`, `data/stations.csv`, `data/designs.json`, `data/tt/`, `data/scenarios.*`.
Outputs: `data/results_05_mi.csv` (or `_s`), images in `data/mi_images/` (or `_s`)

In [ ]:
# --- Imports ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mi_common import MatrixImaging

In [ ]:
# --- Settings ---
PHASE = "p"                       # "p" or "s" (needs the S tables from 03)
SCENARIOS = None                  # None = all, or a list such as ["pos00_dome", "heat_s3"]
DESIGNS = None                    # None = all in designs.json plus "phased (adaptive)"

SUFFIX = "" if PHASE == "p" else "_s"
RESULTS = f"../data/results_05_mi{SUFFIX}.csv"
IMAGE_DIR = f"../data/mi_images{SUFFIX}"

mi = MatrixImaging(PHASE)         # other settings (frequencies, grids, score thresholds) can be passed here
mi

In [ ]:
# --- Run: every scenario x every design. Rerunning skips pairs already in the results file ---
res = mi.run(SCENARIOS, DESIGNS, results=RESULTS, image_dir=IMAGE_DIR)

In [ ]:
# --- Summary: mean scores per design over the scenarios run ---
summary = res.groupby("design")[["coverage", "top_err_km", "crest_err_km", "n_stations", "n_pairs"]].mean()
summary.sort_values("coverage", ascending=False).round(3)

In [ ]:
# --- Scores per design ---
with plt.style.context("default"):
    fig, axs = plt.subplots(1, 3, figsize=(13, 3.6))
    order = summary.sort_values("coverage").index
    for ax, col, lab in zip(axs, ["coverage", "top_err_km", "crest_err_km"],
                            ["Coverage of the BDT top", "Depth error of imaged top (km)", "BDT high location error (km)"]):
        data = [res.loc[res.design == d, col].dropna() for d in order]
        ax.boxplot(data, vert=False, widths=0.5)
        ax.set_yticks(range(1, len(order) + 1), order)
        ax.set_xlabel(lab)
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Images for one scenario: section along the node axis through the scenario centre ---
SHOW = res.scenario.iloc[0]
s, z, secs, true = mi.section(f"{IMAGE_DIR}/{SHOW}.npz", SHOW)

with plt.style.context("default"):
    fig, axs = plt.subplots(1, len(secs), figsize=(3.2 * len(secs), 3.4), sharey=True)
    for ax, (k, sec) in zip(np.atleast_1d(axs), secs.items()):
        ax.pcolormesh(s, z, sec.T, cmap="magma", shading="auto", vmin=0, vmax=1)
        ax.plot(s, true, color="#7fd3ff", ls="--", lw=1.2)
        ax.set(title=k, ylim=(z[-1], z[0]), xlabel="Along node axis (km)")
    np.atleast_1d(axs)[0].set_ylabel("Depth (km)")
    fig.suptitle(SHOW)
    plt.tight_layout()
    plt.show()